# Piece-By-Piece: Walking through Architectural Modifications
We aim to get a drop-and-replace layer-by-layer walk through for architectures. This is inspired by [LoLCats](https://openreview.net/pdf?id=8VtGeyJyx9) and uses our work on guidance.

In [ ]:
import sys
import numpy as np
import torch
import torch.utils.data as data
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from collections import OrderedDict
from tqdm import tqdm

sys.path.append('../neural-architecture-search/imagenet-comparison')
from dataset import ImageNetDataset

device = torch.device('cuda:0')

/storage/vsub851/anaconda3/envs/nas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_dataloaders(batch_size = 256, num_workers = 4):
    train_dataset, val_dataset, test_dataset = ImageNetDataset('train'), ImageNetDataset('validation'), ImageNetDataset('test')
    train_loader = data.DataLoader(train_dataset, batch_size, shuffle = True, num_workers = num_workers, pin_memory = True)
    val_loader = data.DataLoader(val_dataset, batch_size = batch_size, num_workers = num_workers, shuffle = False, pin_memory = True)
    test_loader = data.DataLoader(test_dataset, batch_size = 1)
    return train_loader, val_loader, test_loader

In [3]:
class CKA:
    def __init__(self, device: torch.device):
        self.device = device

    def centering(self, K: torch.Tensor) -> torch.Tensor:
        n = K.size(0)
        I = torch.eye(n, device=self.device)
        H = I - torch.ones((n, n), device=self.device) / n
        return H @ K @ H

    def linear_HSIC(self, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        Lx = X @ X.T
        Ly = Y @ Y.T
        return torch.sum(self.centering(Lx) * self.centering(Ly))

    def linear_CKA(self, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        hsic = self.linear_HSIC(X, Y)
        var1 = torch.sqrt(self.linear_HSIC(X, X))
        var2 = torch.sqrt(self.linear_HSIC(Y, Y))
        return hsic / (var1 * var2 + 1e-12)

## Main Approach
We use ResNet-18 and aim to "linearize" the network. 

* We take a convolutional layer and linearize with an equivalent linear-layer replacement.
* For right now, we only focus on one layer, which we select using an index parameter.
* We optimize only using CKA, minimizing the dissimilarity loss, similar to guidance.
* Then we take the resultant network and finetune

## Network Linearization

To linearize the network, we extract a convolutional layer and replace it with a linear layer. More experiments are needed to tell us *what* convolutional layer should be extracted and replaced or whether **all** of them can be extracted and replaced safely. 

In [4]:
def extract_conv_input_shapes(model, input_size):
    '''
    Extracts input shapes for each convolutional layer across a network. We extract this input 
    shape to specify how large how linear replacement should be.
    '''
    shapes = []
    hooks = []
    def hook(module, inp, out):
        x = inp[0]
        shapes.append(tuple(x.shape[1:]))  # (C, H, W)
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(hook))
    device = next(model.parameters()).device
    dummy = torch.zeros(1, 3, input_size[0], input_size[1], device=device)
    model.eval()
    with torch.no_grad():
        model(dummy)
    for h in hooks:
        h.remove()
    return shapes

In [5]:
class LinearizedConv2d_Old(nn.Module):
    def __init__(self, conv):
        super().__init__()
        assert isinstance(conv, nn.Conv2d)
        self.out_channels = conv.out_channels
        self.kernel_size = conv.kernel_size
        self.stride = conv.stride
        self.padding = conv.padding
        self.dilation = conv.dilation
        self.linear = None

    def forward(self, x):
        B, C, H, W = x.shape
        # compute output spatial dims
        H_out = (H + 2*self.padding[0]
                 - self.dilation[0]*(self.kernel_size[0]-1) - 1)//self.stride[0] + 1
        W_out = (W + 2*self.padding[1]
                 - self.dilation[1]*(self.kernel_size[1]-1) - 1)//self.stride[1] + 1
        flat_dim = C * H * W
        out_dim = self.out_channels * H_out * W_out
        if self.linear is None or self.linear.in_features != flat_dim or self.linear.out_features != out_dim:
            self.linear = nn.Linear(flat_dim, out_dim).to(x.device)
        x_flat = x.view(B, -1)
        y = self.linear(x_flat)
        return y.view(B, self.out_channels, H_out, W_out)

class LinearizedConv2d(nn.Module):
    '''
    Linear Replacement for a 2D Convolutional Layer. Takes an input size and specific convolutional and designs the 
    linear layer with the correct input and correct output shapes. We do have to run a quick forward pass through the conv
    layer to get the correct sizes in this implementation.
    '''
    def __init__(self, conv, input_size):
        super().__init__()
        assert isinstance(conv, nn.Conv2d)
        C_in = conv.in_channels
        H_in, W_in = input_size
        with torch.no_grad():
            dummy = torch.zeros(1, C_in, H_in, W_in).to(device)
            out = conv(dummy)
        _, C_out, H_out, W_out = out.shape
        self.out_channels = C_out
        self.H_out = H_out
        self.W_out = W_out
        flat_dim = C_in * H_in * W_in
        out_dim = C_out * H_out * W_out
        self.linear = nn.Linear(flat_dim, out_dim)
    
    def forward(self, x):
        B = x.size(0)
        x_flat = x.view(B, -1)
        y = self.linear(x_flat)
        return y.view(B, self.out_channels, self.H_out, self.W_out)

## Extracting Convolutions

In [6]:
def list_conv_layers(model):
    convs = []
    for idx, (name, module) in enumerate(model.named_modules(), start = 1):
        if isinstance(module, nn.Conv2d):
            convs.append((idx, name))
    return convs

def replace_nth_conv2d(model, target_idx, input_shape):
    '''
    Replace the nth convolutional layer (given by target_idx)
    Does this recursively because why not?
    '''
    counter = {'i': 0}
    def recurse(mod):
        for name, child in list(mod.named_children()):
            if isinstance(child, nn.Conv2d):
                counter['i'] += 1
                if counter['i'] == target_idx:
                    setattr(mod, name, LinearizedConv2d(child, input_shape))
                    print(f'Replaced Conv2d #{target_idx} at {mod}.{name}')
                    return True
            else:
                if recurse(child):
                    return True
        return False
    if not recurse(model):
        prt_counter = counter['i']
        raise IndexError(f'Only {prt_counter} Conv2d layers; cannot replace #{target_idx}')
    return model

def get_nth_conv(model, n):
    count = 0
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, LinearizedConv2d)):
            count += 1
            if count == n:
                return module
    raise IndexError

## CKA Optimization

In [7]:
def layer_replace(index):
    # Emptying the CUDA cache to start
    torch.cuda.empty_cache()
    orig = torchvision.models.resnet18(pretrained = True).to(device)
    # We're working with ImageNet so assume input shapes of 224x224
    input_shapes = extract_conv_input_shapes(orig, input_size=(224,224))
    C_in, H_in, W_in = input_shapes[index-1]

    # Replacement network
    lin  = torchvision.models.resnet18(pretrained = True).to(device)
    train_loader, _, _ = get_dataloaders(batch_size = 256, num_workers = 12)

    print('Conv2d layers:')
    for idx, name in list_conv_layers(orig):
        print(f'  {idx}: {name}')

    #Replace the nth convolution!
    replace_nth_conv2d(lin, index, (H_in, W_in))
    lin = lin.to(device)
    conv_o = get_nth_conv(orig, index)
    conv_l = get_nth_conv(lin, index)

    # Get activations for optimization of these two layers
    activations_o = {}
    activations_l = {}
    handle_o = conv_o.register_forward_hook(lambda m, i, o: activations_o.setdefault('feat', o))
    handle_l = conv_l.register_forward_hook(lambda m, i, o: activations_l.setdefault('feat', o))

    for p in orig.parameters(): 
        p.requires_grad = False
    # Only tune the linear layers in the linearized ResNet-18 
    for m in lin.modules():
        if not isinstance(m, nn.Linear):
            for p in m.parameters(): 
                p.requires_grad = False
        else:
            for p in m.parameters():
                p.requires_grad = True
    optimizer = optim.AdamW([p for p in lin.parameters() if p.requires_grad], lr=1e-3)
    cka = CKA(device)

    lin.train()
    orig.eval()
    cka_losses = []
    for epoch in range(2):
        epoch_cka_losses = []
        for i, batch in enumerate(tqdm(train_loader, desc = 'Iterating over images...')):
            img = batch['image'].to(device)
            activations_o.clear(); activations_l.clear()
            # Pass through image through and extract activations
            orig(img)
            lin(img)
            feat_o = activations_o['feat']
            feat_l = activations_l['feat']
            B = feat_o.size(0)
            fo_flat = feat_o.view(B, -1)
            fl_flat = feat_l.view(B, -1)
            # Optimize on dissimilarity i.e. 1 - CKA
            loss = 1.0 - cka.linear_CKA(fo_flat, fl_flat)
            if i == 0 and epoch == 0:
                print('Starting CKA...', loss.item())
            optimizer.zero_grad()
            loss.backward() 
            optimizer.step()
            cka_losses.append(loss.item())
            epoch_cka_losses.append(loss.item())
        avg_train_loss = np.mean(epoch_cka_losses)
        print(f'Epoch {epoch}, Avg Loss {avg_train_loss}')
    handle_o.remove()
    handle_l.remove()
    return lin

In [8]:
lin_rn18 = layer_replace(19)

Conv2d layers:
  2: conv1
  8: layer1.0.conv1
  11: layer1.0.conv2
  14: layer1.1.conv1
  17: layer1.1.conv2
  21: layer2.0.conv1
  24: layer2.0.conv2
  27: layer2.0.downsample.0
  30: layer2.1.conv1
  33: layer2.1.conv2
  37: layer3.0.conv1
  40: layer3.0.conv2
  43: layer3.0.downsample.0
  46: layer3.1.conv1
  49: layer3.1.conv2
  53: layer4.0.conv1
  56: layer4.0.conv2
  59: layer4.0.downsample.0
  62: layer4.1.conv1
  65: layer4.1.conv2
Replaced Conv2d #19 at BasicBlock(
  (conv1): LinearizedConv2d(
    (linear): Linear(in_features=25088, out_features=25088, bias=True)
  )
  (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
).conv1


Iterating over images...:   0%|                                                                                                                    | 0/5005 [00:00<?, ?it/s]

Starting CKA... 0.21691203117370605


Iterating over images...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 5005/5005 [10:46<00:00,  7.74it/s]


Epoch 0, Avg Loss 0.02221787114005227


Iterating over images...: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 5005/5005 [10:42<00:00,  7.79it/s]

Epoch 1, Avg Loss 0.007825806329062173


## Model Finetuning

We now design model finetuning to incorporate and reinforce the new layer. I'm not sure how to do this with a linearized ResNet-18 but maybe we'll just try some ImageNet finetuning...

In [9]:
def finetune(lin_rn18):
    train_loader, val_loader, _ = get_dataloaders(batch_size = 8, num_workers = 12)
    for p in lin_rn18.parameters(): 
        p.requires_grad = True
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(lin_rn18.parameters(), lr = 1e-3)
    lin_rn18 = lin_rn18.to(device)
    val_losses = []
    train_losses = []
    for epoch in range(5):
        lin_rn18 = lin_rn18.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc = 'Iterating over validation set...'):
                img, label = batch['image'].to(device), batch['label'].to(device)
                preds = lin_rn18(img)
                loss = loss_fn(preds, label)
                val_loss += loss.item() * img.shape[0]
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)
        print(f'Epoch {epoch + 1}, Validation Loss {avg_val_loss}')
        lin_rn18.train()
        train_loss = 0.0
        for batch in tqdm(train_loader, desc = 'Iterating over training set...'):
            img, label = batch['image'].to(device), batch['label'].to(device)
            preds = lin_rn18(img)
            loss = loss_fn(preds, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_loss += loss.item() * img.shape[0]
        avg_train_loss = train_loss / len(train_loader.dataset)
        print(f'Epoch {epoch + 1}, Training Loss {avg_train_loss}')
    return lin_rn18, val_loader

In [10]:
def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = torch.topk(output, maxk, dim = 1, largest = True, sorted = True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

def eval_loop(model, val_loader, device, aux = False):
    model.eval()
    top1_acc = 0
    top5_acc = 0
    total_samples = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc = 'Iterating over test batches...'):
            img, label = batch['image'].to(device), batch['label'].to(device)
            if not aux:
                outputs = model(img)
            else:
                outputs, _ = model(img)
            acc1, acc5 = accuracy(outputs, label, topk=(1, 5))
            top1_acc += acc1.item() * img.size(0)
            top5_acc += acc5.item() * img.size(0)
            total_samples += img.size(0)
    top1_acc /= total_samples
    top5_acc /= total_samples
    return top1_acc, top5_acc

In [ ]:
lin_rn18, val_loader = finetune(lin_rn18)
top1_acc, top5_acc = eval_loop(lin_rn18, val_loader, device)

Iterating over validation set...: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 6250/6250 [00:56<00:00, 111.22it/s]


Epoch 1, Validation Loss 7.028114251174927


Iterating over training set...:  66%|█████████████████████████████████████████████████████████████▋                               | 106254/160146 [1:34:16<53:58, 16.64it/s]